In [8]:
import pandas as pd

# Update path if needed
df = pd.read_csv("/Users/jaacabrera/Downloads/walmart-sales-dataset-of-45stores.csv", parse_dates=["Date"], low_memory=False)

# Ensure Date is true datetime before using .dt
if "Date" not in df.columns:
    raise KeyError("Date column not found in dataset. Please check the CSV header.")

if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Drop rows with invalid/NaT dates to avoid .dt errors
bad_dates = df["Date"].isna().sum()
if bad_dates > 0:
    print(f"⚠️ Dropping {bad_dates} rows with invalid Date values (could not parse)")
    df = df.dropna(subset=["Date"]).reset_index(drop=True)

# Target
y = df["Weekly_Sales"]

# Basic features (no leakage; do NOT include Weekly_Sales)
# You can start with these raw columns
basic_features = [
    "Holiday_Flag", "Temperature", "Fuel_Price", "CPI", "Unemployment"
]
# Encode store id as numeric feature if present
if "Store" in df.columns:
    df["Store_Encoded"] = df["Store"]
    basic_features.append("Store_Encoded")

# Date-derived features
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek
# pandas >= 1.1: isocalendar() returns a DataFrame-like accessor; ensure int dtype
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Date"].dt.quarter
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)
basic_features += ["Month", "DayOfWeek", "Week", "Quarter", "IsWeekend"]

# Build X
X = df[basic_features].copy()
print("X shape:", X.shape, "| y shape:", y.shape)
print("X columns:", list(X.columns))

⚠️ Dropping 3870 rows with invalid Date values (could not parse)
X shape: (2565, 11) | y shape: (2565,)
X columns: ['Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Store_Encoded', 'Month', 'DayOfWeek', 'Week', 'Quarter', 'IsWeekend']


# Quick Regression Baseline (60% validation, 20% train, 20% test)
This section trains a few simple regression models on the `X` and `y` you built and reports metrics on both validation and test sets.

- Split strategy:
  - Stage 1: Hold out 20% as final test set
  - Stage 2: From the remaining 80%, take 25% for training (overall 20%) and 75% for validation (overall 60%)
- Models: Linear Regression, Ridge (L2), Random Forest
- Metrics: R² (higher better), RMSE, MAE (lower better)

Run this after the first cell that builds `X` and `y`.

In [9]:
# Train/val/test split and baseline models (60/20/20)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Reproducibility
np.random.seed(42)

# Ensure X and y exist from the previous cell
assert 'X' in globals() and 'y' in globals(), "Please run the cell that builds X and y first."

# Stage 1: 20% test holdout
X_rem, X_test, y_rem, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Stage 2: From remaining 80%, take 25% as train (overall 20%) and 75% as validation (overall 60%)
X_train, X_val, y_train, y_val = train_test_split(
    X_rem, y_rem, train_size=0.25, random_state=42
)

print(f"Split sizes -> train: {X_train.shape[0]} ({len(y_train)/len(y):.0%}), val: {X_val.shape[0]} ({len(y_val)/len(y):.0%}), test: {X_test.shape[0]} ({len(y_test)/len(y):.0%})")

# Helper to evaluate on both validation and test

def evaluate(model, name):
    model.fit(X_train, y_train)
    preds_val = model.predict(X_val)
    preds_test = model.predict(X_test)
    r2_v  = r2_score(y_val, preds_val)
    # Older scikit-learn versions don't support squared=False; compute RMSE manually
    rmse_v = np.sqrt(mean_squared_error(y_val, preds_val))
    mae_v = mean_absolute_error(y_val, preds_val)
    r2_t  = r2_score(y_test, preds_test)
    rmse_t = np.sqrt(mean_squared_error(y_test, preds_test))
    mae_t = mean_absolute_error(y_test, preds_test)
    print(
        f"{name:>18} | VAL: R2 {r2_v:7.4f}, RMSE {rmse_v:10.3f}, MAE {mae_v:10.3f} "
        f"| TEST: R2 {r2_t:7.4f}, RMSE {rmse_t:10.3f}, MAE {mae_t:10.3f}"
    )

# Linear Regression (impute + scale)
lin_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

# Ridge Regression (impute + scale)
ridge_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

# Random Forest (impute only; trees don't need scaling)
rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

print("Model Performance (validation and test):")
evaluate(lin_pipe,   "LinearRegression")
evaluate(ridge_pipe, "Ridge(alpha=1)")
evaluate(rf_pipe,    "RandomForest(200)")

Split sizes -> train: 513 (20%), val: 1539 (60%), test: 513 (20%)
Model Performance (validation and test):
  LinearRegression | VAL: R2  0.1043, RMSE 525201.919, MAE 440802.237 | TEST: R2  0.1352, RMSE 530526.030, MAE 447164.433
    Ridge(alpha=1) | VAL: R2  0.1046, RMSE 525119.809, MAE 440753.853 | TEST: R2  0.1351, RMSE 530579.587, MAE 447250.431
 RandomForest(200) | VAL: R2  0.9263, RMSE 150646.206, MAE 106250.339 | TEST: R2  0.9216, RMSE 159722.464, MAE 108559.047


In [10]:
# Optional: Feature importance for Random Forest

try:

    # Refit RF on full training set for importance extraction

    rf_pipe.fit(X_train, y_train)

    rf_model = rf_pipe.named_steps["model"]

    importances = rf_model.feature_importances_

    cols = list(X.columns)

    ranked = sorted(zip(cols, importances), key=lambda t: t[1], reverse=True)

    print("Top feature importances (RandomForest):")

    for name, val in ranked:

        print(f"  {name:>15}: {val:6.3f}")

except Exception as e:

    print("(Skipped feature importances)", e)

Top feature importances (RandomForest):
    Store_Encoded:  0.628
              CPI:  0.178
     Unemployment:  0.095
       Fuel_Price:  0.046
      Temperature:  0.032
             Week:  0.007
        DayOfWeek:  0.007
            Month:  0.004
          Quarter:  0.001
     Holiday_Flag:  0.001
        IsWeekend:  0.001


# Investigate dropped Date rows
This optional cell reloads the CSV with the Date column as raw strings to show which values failed to parse and how many rows were dropped. Use it to decide whether to adjust parsing (e.g., specify a format or day-first).

In [11]:
# Inspect raw Date values that failed to parse
import pandas as pd
from pathlib import Path

# Reuse the same path used above (edit if needed)
csv_path = Path("/Users/jaacabrera/Downloads/walmart-sales-dataset-of-45stores.csv")

raw = pd.read_csv(csv_path, dtype={"Date": "string"}, low_memory=False)
mask = pd.to_datetime(raw["Date"], errors="coerce").isna()
print(f"Total rows: {len(raw)} | Invalid Date rows: {mask.sum()}")

print("\nTop invalid 'Date' values (up to 20):")
print(raw.loc[mask, "Date"].value_counts(dropna=False).head(20))

print("\nSample problematic rows:")
display(raw.loc[mask].head(5))

# Hints to fix parsing:
print("\nIf most invalid values look like dd/mm/yyyy, try dayfirst=True.")
print("If they use a specific format (e.g., 2012-03-15), try format='%Y-%m-%d'.")

Total rows: 6435 | Invalid Date rows: 3870

Top invalid 'Date' values (up to 20):
Date
19-02-2010    45
18-11-2011    45
17-02-2012    45
27-01-2012    45
20-01-2012    45
13-01-2012    45
30-12-2011    45
23-12-2011    45
16-12-2011    45
25-11-2011    45
28-10-2011    45
16-03-2012    45
21-10-2011    45
14-10-2011    45
30-09-2011    45
23-09-2011    45
16-09-2011    45
26-08-2011    45
19-08-2011    45
29-07-2011    45
Name: count, dtype: Int64

Sample problematic rows:


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
6,1,19-03-2010,1472515.79,0,54.58,2.720,211.215635,8.106
7,1,26-03-2010,1404429.92,0,51.45,2.732,211.018042,8.106
10,1,16-04-2010,1466058.28,0,66.32,2.808,210.488700,7.808



If most invalid values look like dd/mm/yyyy, try dayfirst=True.
If they use a specific format (e.g., 2012-03-15), try format='%Y-%m-%d'.
